# Mining Knowledge from Support Tickets
## The Real Problem — and How Clustering Solves It

Support ticket systems have a structural problem.

When an advisor fixes a customer's issue, they update the ticket status to "Closed" and move on.  
They don't write down what they did. They don't link it to similar tickets. They don't explain why the fix worked.

**The knowledge lives in the advisor's head — not in the system.**

This notebook uses 270 real TrakCare support tickets (anonymized) to show:
1. How bad the data quality gap actually is
2. Why clustering works anyway — because it operates on *problem descriptions*, not resolutions
3. How you extract the resolution pattern from the minority that *do* have solution text
4. How that becomes a KB article

---
*Requirements: `pip install sentence-transformers hdbscan pandas openai`*  
*LLM optional — clustering and pattern extraction work without an API key*

## Step 1 — The Data Quality Problem

In [ ]:
import json, pandas as pd, re
from pathlib import Path
from collections import Counter

with open('../data/iservice_demo_tickets.json') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)

print(f"Tickets loaded: {len(df)}")
print()
print(f"{'Category':<30} {'Total':>6} {'Has Solution':>13} {'Gap %':>7}")
print('-' * 60)

cats = df.groupby('Classification').agg(
    total=('ticket_id', 'count'),
    solved=('has_solution', 'sum')
).sort_values('total', ascending=False)

for cat, row in cats.iterrows():
    gap = (1 - row['solved']/row['total']) * 100
    bar = '░' * int(gap / 5)
    print(f"  {cat:<28} {row['total']:>6} {row['solved']:>13} {gap:>6.0f}%  {bar}")

total_solved = df['has_solution'].sum()
total_gap = len(df) - total_solved
print()
print(f"  {'TOTAL':<28} {len(df):>6} {total_solved:>13} {total_gap/len(df)*100:>6.0f}%")
print()
print(f"  {total_gap} of {len(df)} tickets ({total_gap/len(df)*100:.0f}%) have NO resolution text.")
print()
print("This isn't a data quality failure. It's the norm.")
print("Tickets are closed in the system. The fix happens on a call or over email.")
print("Nobody writes it back. The knowledge is lost.")

### What does a ticket with NO resolution look like?

Let's read some of these tickets. The problem is described. The solution is not.

In [ ]:
no_solution = df[~df['has_solution']]

print(f"Sampling {min(4, len(no_solution))} tickets WITHOUT resolution text:")
print()
for _, row in no_solution.sample(min(4, len(no_solution)), random_state=42).iterrows():
    print(f"  {'─'*60}")
    print(f"  {row['ticket_id']} | {row['Classification']} | {row['Status']}")
    print(f"  Problem: {str(row['Problem'])[:200]}")
    print(f"  Solution: (empty)")
    print()

### And what about the minority that DO have solution text?

In [ ]:
with_solution = df[df['has_solution']]

print(f"Sampling {min(3, len(with_solution))} tickets WITH resolution text:")
print()
for _, row in with_solution.sample(min(3, len(with_solution)), random_state=7).iterrows():
    print(f"  {'─'*60}")
    print(f"  {row['ticket_id']} | {row['Classification']}")
    print(f"  Problem: {str(row['Problem'])[:150]}")
    print(f"  Solution: {str(row['Solution'])[:200]}")
    print()

print(f"This is {len(with_solution)} of {len(df)} tickets.")
print(f"The question: can these {len(with_solution)} help the other {len(df)-len(with_solution)}?")

---
## Step 2 — Embed and Cluster

Here's the key insight: **clustering doesn't need resolution text**.

We embed the *problem descriptions* — what the advisor observed, what the customer reported.  
Tickets about the same underlying issue will cluster together regardless of whether they have a solution.

Then we use the minority-with-solutions to suggest fixes for the majority-without.

In [ ]:
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN

print("Loading all-MiniLM-L6-v2 (local, no API key needed)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Use FullContent (Summary + Problem) — NOT Solution
# This is the point: we cluster on what we ALWAYS have
texts = df['FullContent'].fillna(df['Summary']).tolist()
print(f"Embedding {len(texts)} ticket problems (ignoring resolution text)...")

embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
print(f"Done. Shape: {embeddings.shape}")
print()
print("Key: we embedded PROBLEM descriptions only.")
print("Resolution text plays no role in clustering.")
print("Semantic similarity of problems drives the grouping.")

In [ ]:
print("Running HDBSCAN...")
clusterer = HDBSCAN(min_cluster_size=10, metric='euclidean')
labels = clusterer.fit_predict(embeddings)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = sum(1 for l in labels if l == -1)

print(f"Found {n_clusters} clusters, {noise} noise points")
print()

df['cluster'] = labels

for cid, count in Counter(labels).most_common():
    if cid == -1:
        print(f"  Noise:        {count:>4} tickets (unassigned)")
    else:
        cluster = df[df['cluster'] == cid]
        cats = cluster['Classification'].value_counts()
        solved = cluster['has_solution'].sum()
        top_cat = cats.index[0] if len(cats) > 0 else '?'
        print(f"  Cluster {cid}:   {count:>4} tickets | dominant: {top_cat:<25} | {solved} with solutions")

---
## Step 3 — Inspect the Clusters

For each cluster, we ask: **what problem are these tickets really about?**

We look at the problem descriptions (all tickets have these),  
then at the solution text (only some do — and those are our anchors).

In [ ]:
# Show the 3 largest clusters
largest_clusters = sorted(
    [c for c in set(labels) if c != -1],
    key=lambda c: sum(1 for l in labels if l == c),
    reverse=True
)[:3]

for cid in largest_clusters:
    cluster = df[df['cluster'] == cid]
    solved = cluster[cluster['has_solution']]
    unsolved = cluster[~cluster['has_solution']]
    
    cats = cluster['Classification'].value_counts()
    
    print(f"{'='*65}")
    print(f"CLUSTER {cid} — {len(cluster)} tickets")
    print(f"  Categories: {dict(cats.head(3))}")
    print(f"  With solution text: {len(solved)}/{len(cluster)} ({len(solved)/len(cluster)*100:.0f}%)")
    print()
    
    print("  Sample problems (NO solution text — the majority):")
    for _, row in unsolved.sample(min(3, len(unsolved)), random_state=1).iterrows():
        print(f"    {row['ticket_id']}: {str(row['Problem'])[:100]}")
    
    if len(solved) > 0:
        print()
        print("  Sample problems WITH solution (the anchor tickets):")
        for _, row in solved.sample(min(2, len(solved)), random_state=1).iterrows():
            print(f"    {row['ticket_id']}: {str(row['Problem'])[:80]}")
            print(f"      → Solution: {str(row['Solution'])[:100]}")
    print()

---
## Step 4 — Resolution-Anchored Discovery

The cluster with solutions tells us what the fix is.  
The cluster without solutions tells us who else needs it.

**1 resolved ticket → many suggestions for unresolved tickets with the same semantic pattern.**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# For each cluster, show the resolution anchors and their similar unresolved tickets
for cid in largest_clusters[:2]:
    cluster = df[df['cluster'] == cid]
    solved = cluster[cluster['has_solution']]
    unsolved = cluster[~cluster['has_solution']]
    
    if len(solved) == 0:
        continue
    
    print(f"{'─'*65}")
    print(f"Cluster {cid} — resolution anchoring")
    print()
    
    # Show the anchor
    anchor = solved.iloc[0]
    anchor_idx = df.index.get_loc(anchor.name)
    anchor_emb = embeddings[anchor_idx:anchor_idx+1]
    
    print(f"  ANCHOR (resolved): {anchor['ticket_id']}")
    print(f"  Problem:  {str(anchor['Problem'])[:120]}")
    print(f"  Solution: {str(anchor['Solution'])[:150]}")
    print()
    
    # Find most similar unresolved tickets in the cluster
    if len(unsolved) > 0:
        unsolved_indices = [df.index.get_loc(i) for i in unsolved.index[:15]]
        unsolved_embs = embeddings[unsolved_indices]
        sims = cosine_similarity(anchor_emb, unsolved_embs)[0]
        
        top_similar = sorted(zip(sims, unsolved.head(15).iterrows()), key=lambda x: -x[0])[:4]
        
        print(f"  SUGGESTED FIX for these similar unresolved tickets:")
        for sim, (_, row) in top_similar:
            print(f"    {row['ticket_id']} (sim={sim:.3f}): {str(row['Problem'])[:90]}")
        
        print(f"  → Apply: {str(anchor['Solution'])[:120]}")
    print()

---
## Step 5 — Generate the KB Article

Take the cluster's anchor tickets and let an LLM write a structured KB article.

The LLM structures what the data already contains — it doesn't invent.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'setup'))
try:
    from embedder import get_llm_client
    llm_client, llm_model = get_llm_client()
except ImportError:
    from openai import OpenAI
    llm_client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
    llm_model = "gpt-4o-mini"

# Use the first large cluster with solutions
for cid in largest_clusters:
    cluster = df[df['cluster'] == cid]
    solved = cluster[cluster['has_solution']]
    if len(solved) >= 2:
        break

if llm_client and len(solved) > 0:
    cat = cluster['Classification'].value_counts().index[0]
    
    context_parts = []
    for _, row in solved.head(4).iterrows():
        context_parts.append(
            f"Ticket {row['ticket_id']}:\n"
            f"Problem: {str(row['Problem'])[:200]}\n"
            f"Solution: {str(row['Solution'])[:200]}"
        )
    
    # Also include problem-only tickets so LLM sees the scope
    for _, row in cluster[~cluster['has_solution']].head(3).iterrows():
        context_parts.append(
            f"Ticket {row['ticket_id']} (unresolved):\n"
            f"Problem: {str(row['Problem'])[:150]}"
        )
    
    resp = llm_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": (
                f"You are a TrakCare EMR KB article writer for {cat} support. "
                "Write a structured KB article: ## Title, ## Affected Versions, "
                "## Problem, ## Root Cause, ## Resolution Steps (numbered), ## Prevention. "
                "Be specific and actionable. Max 280 words."
            )},
            {"role": "user", "content": "Write KB article from these tickets:\n\n" + "\n\n---\n\n".join(context_parts)}
        ],
        max_tokens=380
    )
    
    print(f"KB ARTICLE — {cat.upper()} CLUSTER")
    print("="*65)
    print(resp.choices[0].message.content)
    print()
    print(f"Source: {len(cluster)} tickets | {len(solved)} with solution | {len(cluster)-len(solved)} unresolved suggested")
else:
    print("LLM not configured. Set OPENAI_API_KEY or OPENROUTER_API_KEY.")
    print()
    print("Without LLM, here's the manually extracted pattern:")
    for cid in largest_clusters:
        cluster = df[df['cluster'] == cid]
        solved = cluster[cluster['has_solution']]
        if len(solved) >= 2:
            cat = cluster['Classification'].value_counts().index[0]
            print(f"\nCLUSTER: {cat}")
            for _, row in solved.head(3).iterrows():
                print(f"  Problem:  {str(row['Problem'])[:100]}")
                print(f"  Solution: {str(row['Solution'])[:100]}")
            break

---
## Step 6 — What We Found

Let's be honest about what this technique can and can't do.

In [ ]:
total_solved = df['has_solution'].sum()
total_unsolved = len(df) - total_solved

print("SUMMARY")
print("="*65)
print(f"  Tickets analyzed:          {len(df)}")
print(f"  Clusters found:            {n_clusters}")
print()
print("The data quality problem:")
print(f"  With solution text:        {total_solved} ({total_solved/len(df)*100:.0f}%)")
print(f"  WITHOUT solution text:     {total_unsolved} ({total_unsolved/len(df)*100:.0f}%)")
print()
print("What clustering does:")
print("  ✓ Groups semantically similar tickets — regardless of resolution status")
print("  ✓ Surfaces the {total_solved} resolved tickets as anchors for the rest")
print("  ✓ Identifies which unresolved tickets likely share a root cause")
print()
print("What clustering does NOT do:")
print("  ✗ Invent resolution text for tickets that have none")
print("  ✗ Guarantee that the anchor's fix applies to all cluster members")
print("  ✗ Replace human judgment on whether the fix is appropriate")
print()
print("The value proposition:")
print(f"  A new advisor gets ticket #{df[~df['has_solution']].iloc[0]['ticket_id']}.")
print(f"  Without this tool: they start from zero.")
print(f"  With this tool: they see {len(df[df['has_solution']].head(3))} similar tickets that were resolved,")
print(f"  with the actual fix steps. That's institutional knowledge surfaced automatically.")

---
## Going Further

**More tickets = better clusters.**  
The 270-ticket demo above shows the pattern. At 16,659 embedded tickets, 4 distinct patterns emerged.  
At 818K (the full backlog), the clusters get richer and the resolution anchors more representative.

**The discovery loop** (in `planetcare_system_demo.ipynb`) shows how the knowledge graph guides  
which tickets to embed next — so you don't have to embed 818K at once.

**Adapting to your data:**
1. Export your tickets to JSON: `{ticket_id, Summary, Problem, Solution, Classification, Status}`
2. Replace `data/iservice_demo_tickets.json` with your export  
3. Run this notebook — clustering works on any EMR, any language

The clustering operates on semantic similarity of problem descriptions.  
As long as advisors describe what they see, the model can find the patterns.
